In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/dirty_financial_transactions.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['transaction_date_clean'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
df = df.dropna(subset=['transaction_date_clean']).sort_values('transaction_date_clean').drop(columns=['transaction_date_clean']).reset_index(drop=True)

split = int(len(df) * 0.9)
df_initial = df.iloc[:split].copy()
df_delta = df.iloc[split:].copy()

duplicates = df_initial.sample(n=5, random_state=42).copy()
df_delta = pd.concat([df_delta, duplicates], ignore_index=True)

new_records = pd.DataFrame({
    'transaction_id': ['T_NEW_9991', 'T_NEW_9992'],
    'transaction_date': [pd.Timestamp('2026-08-26'), pd.Timestamp('2026-08-26')],
    'customer_id': ['c9999', 'c8888'],
    'product_name': ['Smartphone', 'Laptop'],
    'quantity': [1.0, 2.0],
    'price': [500.0, 1200.0],
    'payment_method': ['Cash', 'Paypal'],
    'transaction_status': ['completed', 'completed']
})
df_delta = pd.concat([df_delta, new_records], ignore_index=True)

df_delta.iloc[0, df_delta.columns.get_loc('product_name')] = 'Phone'

branches_df = pd.DataFrame({
    'branch': ['Branch A', 'Branch B', 'Branch C', 'Branch D', 'Branch E', 'Branch F'],
    'city': ['Warsaw', 'Berlin', 'Paris', 'Prague', 'Vienna', 'Madrid'],
    'country': ['Poland', 'Germany', 'France', 'Czechia', 'Austria', 'Spain'],
    'region': ['Europe', 'Europe', 'Europe', 'Europe', 'Europe', 'Europe'],
    'department': ['Retail Banking', 'Retail Banking', 'Retail Banking', 'Retail Banking', 'Retail Banking', 'Retail Banking'],
    'manager': ['Anna', 'John', 'Michael', 'Elena', 'Stefan', 'Lucia'],
    'lat': [52.2297, 52.5200, 48.8566, 50.0755, 48.2082, 40.4168],
    'lon': [21.0122, 13.4050, 2.3522, 14.4378, 16.3738, -3.7038]
})

product_branch_mapping = pd.DataFrame({
    'product_name': ['Smartphone', 'Coffee Machine', 'Tablet', 'Laptop', 'Headphones', 'Other'],
    'branch': ['Branch A', 'Branch B', 'Branch C', 'Branch D', 'Branch E', 'Branch F']
})

def standardize_product_name(name):
    if not isinstance(name, str):
        return 'Other'
    val = name.strip().lower()
    if val.startswith(('head', 'headp', 'headph', 'headpho', 'headphon', 'headphone', 'hea', 'he', 'h', 'ear')):
        return 'Headphones'
    elif val.startswith(('smart', 'smartp', 'smartph', 'smartpho', 'smartphon', 'smartphone', 'sma', 'sm', 's', 'phone')):
        return 'Smartphone'
    elif val.startswith(('coffee', 'coffee m', 'coffee ma', 'coffee mac', 'coffee mach', 'coffee machi', 'coffee machin', 'cof', 'coffe', 'co', 'c')):
        return 'Coffee Machine'
    elif val.startswith(('tablet', 'tabl', 'tab', 'ta', 'table', 't')):
        return 'Tablet'
    elif val.startswith(('laptop', 'lapto', 'lapt', 'lap', 'la', 'l')):
        return 'Laptop'
    else:
        return 'Other'

def clean_and_transform(df_in):
    df_out = df_in.copy()
    
    df_out.columns = df_out.columns.str.strip().str.lower().str.replace(' ', '_')
    
    df_out['quantity'] = pd.to_numeric(df_out['quantity'], errors='coerce')
    df_out['price'] = pd.to_numeric(
        df_out['price'].astype(str).str.replace(r'[^0-9.-]', '', regex=True), 
        errors='coerce'
    ).abs()
    
    df_out['total_amount'] = df_out['quantity'] * df_out['price']
    
    if 'product_name' in df_out.columns:
        df_out['product_name'] = df_out['product_name'].apply(standardize_product_name)
    else:
        df_out['product_name'] = 'Other'
        
    geo_cols_to_drop = [c for c in ['branch', 'city', 'country', 'region', 'department', 'manager', 'lat', 'lon'] if c in df_out.columns]
    df_out = df_out.drop(columns=geo_cols_to_drop, errors='ignore')
    
    df_out = pd.merge(df_out, product_branch_mapping, on='product_name', how='left')
    df_out['branch'] = df_out['branch'].fillna('Branch F')
    
    df_out = pd.merge(df_out, branches_df, on='branch', how='left')
    
    df_out['value_tier'] = pd.cut(
        df_out['total_amount'].fillna(0), 
        bins=[-float('inf'), 100, 1000, float('inf')], 
        labels=['Low', 'Medium', 'High']
    ).astype(str)

    df_out.columns = df_out.columns.str.strip().str.lower().str.replace(' ', '_')
    df_out = df_out.loc[:, ~df_out.columns.duplicated()]
    df_out = df_out.replace(['nan', 'NaN', 'NAN', 'None', '<NA>'], np.nan)
    
    return df_out

df_initial = clean_and_transform(df_initial)
df_delta = clean_and_transform(df_delta)

target_customer = df_initial['customer_id'].dropna().iloc[0]
df_delta.loc[df_delta['customer_id'] == target_customer, 'value_tier'] = 'High'

df_initial.to_csv('../data/initial_load.csv', index=False, na_rep='')
df_delta.to_csv('../data/delta_load.csv', index=False, na_rep='')

1. Purpose of the three-layer DWH architecture:

If we connect Power BI straight to raw files, any small change in the file structure will instantly break all dashboards. Also, raw data is very dirty. The DWH layers clean this up and make sure everyone in the company looks at the same verified numbers (a Single Source of Truth). Plus, raw tables are not built for fast analytics, while Mart tables are ready for business use.


2. Why do reporting tools like Power BI perform significantly better with a Star Schema rather than a highly normalized relational model?

In a normalized 3NF model, data is split into many small tables. To get a simple report, the database has to do a lot of heavy JOIN operations between tables.
A Star Schema combines things into wider, flat dimension tables around one central fact table. Power BI's internal engine works much faster with this structure, compresses data better, and makes it much easier to write measures and formulas without needing to handle complex database links.

In [ ]:
df_initial.info()

df_delta.info()